# JPX Tokyo Stock Exchange Prediction

## Описание задания

Это соревнование на платформе Kaggle, организованное Japan Exchange Group (JPX) при поддержке AlpacaJapan Co.,Ltd. Участникам предлагается построить модель для прогнозирования доходности акций японского рынка (включая акции, торгуемые на Токийской фондовой бирже) и сформировать инвестиционный портфель.

**Основная задача:** ранжировать около 2000 акций от ожидаемой наибольшей доходности к наименьшей. Оценка модели производится на основе разницы в реальной доходности между 200 лучшими и 200 худшими акциями в портфеле. Соревнование является "forecasting" – модели будут оцениваться на реальных рыночных данных, собранных после окончания этапа подачи решений.

**Метрика оценки:** Sharpe Ratio (коэффициент Шарпа) ежедневной доходности "длинной-короткой" стратегии (покупка 200 лучших акций и короткая продажа 200 худших).

> **Важно про формат ответа.** Это *code competition* с time-series API. Файл `submission.csv`
> создаёт **сам API** во время итерации по тестовым дням (`env.predict(...)`).
> Вручную записывать `submission.csv` нельзя — Kaggle закрывает этот путь, и ноутбук
> падает с `PermissionError: [Errno 1] Operation not permitted: 'submission.csv'`.
> Ниже в разделе 10 вся выгрузка оформлена через официальный API.

## Описание таблиц (файлов данных)

Данные разделены на несколько файлов и папок, охватывающих различные периоды времени.

### Основные файлы
- `stock_prices.csv` - Основной файл с ежедневными ценами закрытия для каждой акции. Содержит целевую переменную (`Target`).
- `options.csv` - Данные о статусе различных опционов (производных инструментов) на основе более широкого рынка. Могут содержать неявные прогнозы будущей цены.
- `secondary_stock_prices.csv` - Данные по менее ликвидным ценным бумагам (не входящим в основные 2000 акций). Не оцениваются напрямую, но могут быть полезны для анализа рынка в целом.
- `trades.csv` - Сводная статистика по объемам торгов за предыдущую торговую неделю.
- `financials.csv` - Результаты квартальных отчетов компаний.
- `stock_list.csv` - Справочная таблица: соответствие между кодом ценной бумаги (`SecuritiesCode`) и названием компании, а также информация об отрасли.

### Папки с данными
- `train_files/` - Данные за основной период обучения.
- `supplemental_files/` - Данные за динамический период, которые дополняют тренировочный набор. Обновляются несколько раз в течение соревнования, а также в начале фазы прогнозирования.
- `example_test_files/` - Данные за публичный тестовый период, предназначенные для офлайн-тестирования. Структура соответствует тому, что выдает API (включает пример файла `sample_submission`).

### Примечания по данным
В файлах, полученных через API (например, в `example_test_files`), отсутствует колонка `Target`. Её можно рассчитать на основе колонки `Close`.

Целевая переменная `Target` представляет собой доходность от покупки акции на следующий день и продажи через день после этого.

Все данные предоставляются в формате `CSV`.

## Импорт необходимых библиотек

In [ ]:
import os
import sys
import gc
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import pyplot as plt
#from IPython.core.display import display, HTML

# Статистические тесты и линейный бейзлайн
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Градиентный бустинг. LightGBM есть и на Kaggle, и локально; если его нет —
# используем HistGradientBoosting, а если нет и его — обычный GradientBoosting.
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    lgb = None
    LGBM_AVAILABLE = False

try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    HISTGB_AVAILABLE = True
except ImportError:
    HISTGB_AVAILABLE = False

from sklearn.ensemble import GradientBoostingRegressor

# Настройка отображения графиков
%matplotlib inline

# Игнорируем предупреждения для чистоты вывода
warnings.filterwarnings("ignore")

# Имя стиля matplotlib зависит от версии: в старых образах (Python 3.7, matplotlib < 3.6)
# это 'seaborn-darkgrid', в новых — 'seaborn-v0_8-darkgrid'. Подбираем то, что есть.
for _style in ('seaborn-v0_8-darkgrid', 'seaborn-darkgrid', 'ggplot', 'default'):
    try:
        plt.style.use(_style)
        break
    except OSError:
        continue
sns.set_palette("husl")

print("Python:", sys.version.split()[0], "|", sys.platform)
print("pandas:", pd.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("LightGBM доступен:", LGBM_AVAILABLE)

## 0. Вспомогательные функции

Здесь собраны три блока:

1. функции понижения разрядности (экономия памяти на 2+ млн строк);
2. **метрика соревнования** (Sharpe Ratio спреда доходностей) — та же логика, что использует Kaggle;
3. **feature engineering**: лаги, скользящие окна, свечные паттерны и рыночные (кросс-секционные) признаки.

> **Важная правка по сравнению с первой версией ноутбука.** Лаги и скользящие средние
> обязательно считаются **внутри одной акции** (`groupby('SecuritiesCode')`). В прежней
> версии `shift()` / `rolling()` применялись ко всему датасету целиком, поэтому «предыдущая
> строка» принадлежала другой бумаге — это искажало признаки и подмешивало данные разных
> инструментов друг в друга. Здесь такого больше нет.

In [ ]:
# ============================== Память =====================================
def downcast_df_int_columns(df):
    """
    Уменьшает занимаемую память для целочисленных колонок DataFrame.
    Преобразует int64 -> int32 или меньше.
    """
    list_of_columns = list(df.select_dtypes(include=["int32", "int64"]).columns)

    if len(list_of_columns) >= 1:
        max_string_length = max([len(col) for col in list_of_columns])
        print("Понижение разрядности целочисленных колонок:", list_of_columns, "\n")

        for col in list_of_columns:
            print(f"  {col.ljust(max_string_length + 2)[:max_string_length + 2]} "
                  f"было {str(round(df[col].memory_usage(deep=True) * 1e-6, 2)).rjust(8)} МБ -> ", end="")
            df[col] = pd.to_numeric(df[col], downcast="integer")
            print(f"{str(round(df[col].memory_usage(deep=True) * 1e-6, 2)).rjust(8)} МБ")
    else:
        print("Нет целочисленных колонок для понижения разрядности")

    gc.collect()
    print("Готово!\n")


def downcast_df_float_columns(df):
    """
    Уменьшает занимаемую память для вещественных колонок DataFrame.
    Преобразует float64 -> float32.
    """
    list_of_columns = list(df.select_dtypes(include=["float64"]).columns)

    if len(list_of_columns) >= 1:
        max_string_length = max([len(col) for col in list_of_columns])
        print("Понижение разрядности вещественных колонок:", list_of_columns, "\n")

        for col in list_of_columns:
            print(f"  {col.ljust(max_string_length + 2)[:max_string_length + 2]} "
                  f"было {str(round(df[col].memory_usage(deep=True) * 1e-6, 2)).rjust(8)} МБ -> ", end="")
            df[col] = pd.to_numeric(df[col], downcast="float")
            print(f"{str(round(df[col].memory_usage(deep=True) * 1e-6, 2)).rjust(8)} МБ")
    else:
        print("Нет вещественных колонок для понижения разрядности")

    gc.collect()
    print("Готово!\n")


# ====================== Скользящие окна внутри бумаги =======================
def flatten_group_index(series):
    """
    Приводит результат groupby-агрегации к исходному индексу df.

    В разных версиях pandas groupby.rolling()/ewm() возвращают либо обычный
    индекс, либо MultiIndex (ключ группы, исходный индекс) — приводим к общему виду.
    """
    if isinstance(series.index, pd.MultiIndex):
        return series.reset_index(level=0, drop=True)
    return series


def grouped_rolling(df, col, window, how="mean", min_periods=None):
    """
    Скользящая статистика по колонке `col`, посчитанная ОТДЕЛЬНО для каждой акции.

    Возвращает Series с тем же индексом, что и df, поэтому её можно напрямую
    записать в новую колонку.
    """
    if min_periods is None:
        min_periods = max(2, window // 4)
    rolled = df.groupby("SecuritiesCode", sort=False)[col].rolling(window, min_periods=min_periods)
    return flatten_group_index(getattr(rolled, how)())


# ============================== Метрика =====================================
def calc_spread_return_per_day(day_df, portfolio_size=200, toprank_weight_ratio=2):
    """
    Спред-доходность портфеля за один день (официальная реализация JPX).

    day_df — строки одного дня с колонками Rank (плотные 0..N-1) и Target.
    Покупаем портфель из `portfolio_size` акций с лучшим Rank и одновременно продаём
    в шорт портфель из `portfolio_size` худших. Внутри каждого портфеля веса линейно
    убывают: вес лучшей акции в `toprank_weight_ratio` раз больше веса худшей.

    Если бумаг в дне меньше, чем два портфеля, день пропускается (возвращается NaN) —
    на скрытых данных такое может встретиться, и ронять весь ноутбук не нужно.
    """
    if len(day_df) < 2 * portfolio_size:
        return np.nan

    ordered = np.sort(day_df['Rank'].to_numpy())
    assert ordered[0] == 0 and ordered[-1] == len(ordered) - 1, "Rank должен быть плотным (0..N-1)"

    weights = np.linspace(start=toprank_weight_ratio, stop=1, num=portfolio_size)
    targets = day_df.sort_values('Rank')['Target'].to_numpy()
    purchase = (targets[:portfolio_size] * weights).sum() / weights.mean()
    short = (targets[::-1][:portfolio_size] * weights).sum() / weights.mean()
    return purchase - short


def calc_spread_return_sharpe(df, portfolio_size=200, toprank_weight_ratio=2):
    """
    Sharpe Ratio = среднее дневных спред-доходностей / их стандартное отклонение.
    Возвращает (sharpe, Series спред-доходностей по дням) — Series пригодится для тестов.
    """
    raw = {day: calc_spread_return_per_day(day_df, portfolio_size, toprank_weight_ratio)
           for day, day_df in df.groupby('Date', sort=True)}
    buf = pd.Series({day: value for day, value in raw.items() if np.isfinite(value)},
                    dtype='float64').sort_index()
    if buf.empty:
        return np.nan, buf
    return buf.mean() / buf.std(), buf


# ========================= Feature engineering ==============================
def add_features(df):
    """
    Создаёт признаки для каждой акции по её собственной истории.

    Ограничения (важны и для честной валидации, и для работы через API):
    * все лаги и скользящие окна считаются внутри одного `SecuritiesCode`;
    * ни один признак не использует данные из будущего;
    * достаточно колонок Date, SecuritiesCode, Open, High, Low, Close, Volume,
      ExpectedDividend, SupervisionFlag.
    """
    df = df.sort_values(['SecuritiesCode', 'Date']).reset_index(drop=True)
    grp = df.groupby('SecuritiesCode', sort=False)

    close, open_, high, low = df['Close'], df['Open'], df['High'], df['Low']
    prev_close = grp['Close'].shift(1)

    # --- Лаги и доходности --------------------------------------------------
    df['ret_1'] = close / prev_close - 1
    df['ret_5'] = close / grp['Close'].shift(5) - 1
    df['ret_20'] = close / grp['Close'].shift(20) - 1
    df['ret_60'] = close / grp['Close'].shift(60) - 1
    df['gap'] = open_ / prev_close - 1

    # --- Волатильность ------------------------------------------------------
    df['vol_20'] = grouped_rolling(df, 'ret_1', 20, 'std', min_periods=10)
    df['vol_60'] = grouped_rolling(df, 'ret_1', 60, 'std', min_periods=30)

    # --- Цена относительно скользящих средних --------------------------------
    sma20 = grouped_rolling(df, 'Close', 20, 'mean', min_periods=10)
    sma60 = grouped_rolling(df, 'Close', 60, 'mean', min_periods=30)
    ema20 = flatten_group_index(grp['Close'].ewm(span=20, adjust=False).mean())
    df['px_sma20'] = close / sma20 - 1
    df['px_sma60'] = close / sma60 - 1
    df['px_ema20'] = close / ema20 - 1

    # --- Свечной паттерн -----------------------------------------------------
    df['body'] = (close - open_) / open_
    df['hl_range'] = (high - low) / close
    df['upper_shadow'] = (high - np.maximum(close, open_)) / close
    df['lower_shadow'] = (np.minimum(close, open_) - low) / close

    # --- Объём ---------------------------------------------------------------
    log_vol = np.log1p(df['Volume'].clip(lower=0).fillna(0).astype('float64'))
    df['log_vol'] = log_vol
    vol_ma = grouped_rolling(df, 'log_vol', 20, 'mean', min_periods=10)
    vol_sd = grouped_rolling(df, 'log_vol', 20, 'std', min_periods=10)
    df['vol_z'] = (log_vol - vol_ma) / vol_sd

    # --- Календарь и флаги ----------------------------------------------------
    df['dow'] = df['Date'].dt.dayofweek.astype('int8')
    df['month'] = df['Date'].dt.month.astype('int8')
    df['has_dividend'] = df['ExpectedDividend'].notna().astype('int8')
    df['supervision'] = df['SupervisionFlag'].fillna(False).astype(bool).astype('int8')

    # --- Рыночные (кросс-секционные) признаки ---------------------------------
    df['mkt_ret_1'] = df.groupby('Date')['ret_1'].transform('mean')
    df['excess_ret_1'] = df['ret_1'] - df['mkt_ret_1']
    df['cs_rank_ret1'] = df.groupby('Date')['ret_1'].rank(pct=True)
    df['cs_rank_vol20'] = df.groupby('Date')['vol_20'].rank(pct=True)

    # --- Мягкое ограничение выбросов ------------------------------------------
    # Первые дни истории дают нестабильные отношения к скользящим средним,
    # а единичные события — экстремальные доходности. Подрезаем хвосты:
    # это устойчивее и для линейной модели, и для бустинга.
    ratio_clip = {'ret_1': 0.5, 'ret_5': 0.8, 'ret_20': 1.0, 'ret_60': 1.5,
                  'gap': 0.5, 'body': 0.5, 'hl_range': 0.5,
                  'px_sma20': 0.5, 'px_sma60': 0.7, 'px_ema20': 0.5,
                  'vol_z': 10.0}
    for col, limit in ratio_clip.items():
        df[col] = df[col].clip(-limit, limit)

    return df


# Список итоговых признаков, которые уходят в модель
FEATURES = [
    'ret_1', 'ret_5', 'ret_20', 'ret_60', 'gap',
    'vol_20', 'vol_60', 'vol_z',
    'px_sma20', 'px_sma60', 'px_ema20',
    'body', 'hl_range', 'upper_shadow', 'lower_shadow',
    'mkt_ret_1', 'excess_ret_1', 'cs_rank_ret1', 'cs_rank_vol20',
    'dow', 'month', 'has_dividend', 'supervision',
]
TARGET = 'Target'
PORTFOLIO_SIZE = 200

# Размер скользящего окна истории, которое доносится день за днём в цикле API
HISTORY_DAYS = 150


# ============================ Пути к данным =================================
def find_data_dir():
    """
    Ищем корень с данными соревнования (папку, где лежит train_files/stock_prices.csv).

    Локально это текущая папка ноутбука или её родитель. На Kaggle имя каталога в
    /kaggle/input произвольное (jpx-tokyo-stock-exchange-prediction, -1, -2 и т.п.),
    поэтому там ищем рекурсивно, а не поhardcoded имени.
    """
    # 1) Каталоги рядом с ноутбуком + типовые точки монтирования
    for path in ['.', '..', '../..', '/kaggle/input', '/input']:
        if os.path.exists(os.path.join(path, 'train_files', 'stock_prices.csv')):
            return path

    # 2) Рекурсивный поиск во входных каталогах Kaggle (в любом месте и на любой глубине)
    for root in ['/kaggle/input', '/input']:
        if not os.path.isdir(root):
            continue
        for dirpath, _dirnames, _filenames in os.walk(root):
            if os.path.exists(os.path.join(dirpath, 'train_files', 'stock_prices.csv')):
                return dirpath

    # 3) Ничего не нашли — показываем, что реально есть во входных каталогах
    listing = []
    for root in ['/kaggle/input', '/input', '.']:
        if os.path.isdir(root):
            listing.append(f"{root} -> {sorted(os.listdir(root))[:20]}")
    raise FileNotFoundError(
        "Не найден каталог с train_files/stock_prices.csv.\n"
        "В ноутбуке НЕ прикреплены данные соревнования. Что нужно сделать:\n"
        "  1) открыть ноутбук в Kaggle;\n"
        "  2) Add data -> Competition data -> jpx-tokyo-stock-exchange-prediction -> Add;\n"
        "  3) запустить ноутбук заново (Run all) с чистого состояния.\n"
        "Сейчас во входных каталогах:\n  " + "\n  ".join(listing)
    )


DATA_DIR = find_data_dir()
print("Каталог с данными:", os.path.abspath(DATA_DIR))


# ------------------ Официальный time-series API соревнования -----------------
# Пакет jpx_tokyo_market_prediction лежит прямо в каталоге с данными,
# поэтому сначала добавляем этот каталог в sys.path, а потом импортируем.
#
# ВАЖНО: пакет собран под CPython 3.7 (файл competition.cpython-37m-x86_64-linux-gnu.so),
# поэтому он импортируется ТОЛЬКО в окружении Python 3.7. В более новом Python импорт
# падает, и submission.csv создать невозможно: файл формирует именно API.
API_IMPORT_ERROR = None
for _extra_path in (os.path.abspath(DATA_DIR), DATA_DIR):
    if _extra_path not in sys.path:
        sys.path.insert(0, _extra_path)

try:
    import jpx_tokyo_market_prediction
    API_AVAILABLE = True
except Exception as exc:            # ImportError / ModuleNotFoundError
    jpx_tokyo_market_prediction = None
    API_AVAILABLE = False
    API_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"

ON_KAGGLE = os.path.isdir('/kaggle/input')

print("Python:", sys.version.split()[0])
print("Официальный API доступен:", API_AVAILABLE)
if not API_AVAILABLE:
    print("Ошибка импорта API:", API_IMPORT_ERROR)

if ON_KAGGLE and not API_AVAILABLE:
    print("\n" + "!" * 78)
    print("!! ВНИМАНИЕ: time-series API не подключился — submission.csv не будет создан.")
    print(f"!! Нужен Python 3.7 (сейчас {sys.version.split()[0]}): пакет собран под CPython 3.7.")
    print("!! Ноутбук нужно создать со страницы соревнования: Code -> New Notebook")
    print("!! (это Competition Notebook с нужным окружением), а не загружать .ipynb.")
    print("!" * 78 + "\n")

## 1. Загрузка и первичный анализ данных

In [ ]:
# Загружаем данные
data = pd.read_csv(os.path.join(DATA_DIR, 'train_files', 'stock_prices.csv'))
data['Date'] = pd.to_datetime(data['Date'], format='%Y-%m-%d')

print(f"Размер данных: {data.shape}")
print(f"Количество уникальных RowId: {data['RowId'].nunique():,}")
print(f"Количество уникальных дат: {data['Date'].nunique():,}")
print(f"Количество уникальных ценных бумаг: {data['SecuritiesCode'].nunique():,}")
print("\n")

## 2. Разведочный анализ данных (EDA)

In [ ]:
# Общий вид датасета
data.head()

In [ ]:
# Информация о данных до оптимизации
print("\n--- Информация о данных ДО оптимизации памяти ---")
data.info(memory_usage='deep')

In [ ]:
# График "Распределения цен закрытия"
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма
axes[0].hist(data['Close'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Распределение цен закрытия')
axes[0].set_xlabel('Цена закрытия')
axes[0].set_ylabel('Частота')

# Boxplot для выявления выбросов
try:
    axes[1].boxplot(data['Close'].dropna(), vert=True)
except TypeError:
    axes[1].boxplot(data['Close'].dropna(), orientation='vertical')
axes[1].set_title('Boxplot цен закрытия')
axes[1].set_ylabel('Цена закрытия')

plt.tight_layout()
plt.show()

In [ ]:
# График "Динамика цен по времени (средние значения)"
daily_avg = data.groupby('Date')[['Open', 'High', 'Low', 'Close']].mean()

fig, ax = plt.subplots(figsize=(15, 6))
daily_avg.plot(ax=ax)
ax.set_title('Средние цены по дням (Open, High, Low, Close)')
ax.set_xlabel('Дата')
ax.set_ylabel('Средняя цена')
ax.legend(['Open', 'High', 'Low', 'Close'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# График "Корреляционная матрица"
numeric_cols = data.select_dtypes(include=[np.number]).columns
corr_matrix = data[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='coolwarm', square=True, linewidths=0.5,
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Корреляционная матрица числовых признаков')
plt.tight_layout()
plt.show()

In [ ]:
# График "Объем торгов по времени"
daily_volume = data.groupby('Date')['Volume'].sum()

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(daily_volume.index, daily_volume.values, color='green', alpha=0.7)
ax.set_title('Суммарный дневной объем торгов')
ax.set_xlabel('Дата')
ax.set_ylabel('Объем торгов')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# График "Распределение по месяцам"
data['Month_num'] = data['Date'].dt.month
monthly_counts = data['Month_num'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(monthly_counts.index, monthly_counts.values, color='skyblue', edgecolor='black')
ax.set_title('Количество записей по месяцам')
ax.set_xlabel('Месяц')
ax.set_ylabel('Количество записей')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 
                    'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек'])
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# График "Выбросы по цене закрытия (Z-score)"
close_mean = data['Close'].mean()
close_std = data['Close'].std()
z_scores = (data['Close'] - close_mean) / close_std
outliers = data[abs(z_scores) > 3]

fig, ax = plt.subplots(figsize=(14, 5))

# Scatter plot для визуализации выбросов
ax.scatter(data.index, data['Close'], alpha=0.3, s=1, label='Нормальные значения', color='blue')
ax.scatter(outliers.index, outliers['Close'], color='red', s=5, label='Выбросы (|Z| > 3)')
ax.set_title(f'Выбросы по цене закрытия (всего: {len(outliers)} записей)')
ax.set_xlabel('Индекс записи')
ax.set_ylabel('Цена закрытия')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Количество выбросов: {len(outliers):,} ({len(outliers)/len(data)*100:.4f}% от всех данных)")

## 3. Оптимизация памяти

In [ ]:
# Сохраняем размер до оптимизации
mem_before = data.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Размер данных ДО оптимизации: {mem_before:.2f} МБ")

# Понижаем разрядность целочисленных колонок
downcast_df_int_columns(data)

# Понижаем разрядность вещественных колонок
downcast_df_float_columns(data)

# Информация после оптимизации
print("\n--- Информация о данных ПОСЛЕ оптимизации памяти ---")
data.info(memory_usage='deep')

mem_after = data.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"\nРазмер данных ПОСЛЕ оптимизации: {mem_after:.2f} МБ")
print(f"Сэкономлено: {mem_before - mem_after:.2f} МБ ({(1 - mem_after/mem_before)*100:.1f}%)")

## 4. Данные для обучения и схема валидации

Для обучения используем **всё доступное прошлое**: `train_files/stock_prices.csv` плюс
`supplemental_files/stock_prices.csv`. Дополнительные данные обновляются организаторами и
к моменту старта фазы прогнозирования содержат последний известный торговый день — тестовые
данные идут строго после него.

**Схема валидации — walk-forward по времени.** Данные делятся НЕ случайно, а по датам:

```
|--------------------- обучение (TRAIN_DAYS дней) -----|-- фолд 1 --|
                                                   |-- фолд 2 --|
                                                                  ...
```

Так проверяется ровно тот сценарий, который будет на реальной бирже: модель обучена на
прошлом, а оценивается на будущем. Случайное перемешивание здесь давало бы завышенную оценку
и, что хуже, позволяло бы «подсмотреть» будущее в признаках.

In [ ]:
DTYPES = {
    'RowId': 'category',
    'Date': 'string',
    'SecuritiesCode': 'int32',
    'Open': 'float32',
    'High': 'float32',
    'Low': 'float32',
    'Close': 'float32',
    'Volume': 'float32',
    'ExpectedDividend': 'float32',
    'SupervisionFlag': 'object',
    'AdjustmentFactor': 'float32',
    'Target': 'float32',
}


def read_prices_csv(path):
    """Читаем stock_prices.csv сразу с узкими типами — это заметно экономит память."""
    header = pd.read_csv(path, nrows=0).columns
    dtypes = {c: DTYPES[c] for c in header if c in DTYPES}
    df = pd.read_csv(path, dtype=dtypes)
    df['Date'] = pd.to_datetime(df['Date'])
    print(f"{path}: {df.shape[0]:,} строк | {df['Date'].nunique()} дней | "
          f"{df['SecuritiesCode'].nunique()} бумаг | {df['Date'].min().date()} — {df['Date'].max().date()}")
    return df


# DATA_DIR определён в разделе 0 — он одинаково работает и локально, и на Kaggle
# EDA-фрейм больше не нужен — освобождаем память перед обучением
del data
gc.collect()

train_prices = read_prices_csv(os.path.join(DATA_DIR, 'train_files/stock_prices.csv'))

supp_path = os.path.join(DATA_DIR, 'supplemental_files', 'stock_prices.csv')
supp_prices = read_prices_csv(supp_path) if os.path.exists(supp_path) else None

if supp_prices is not None:
    prices = pd.concat([train_prices, supp_prices], ignore_index=True)
    del train_prices, supp_prices
else:
    prices = train_prices

prices = prices.sort_values(['SecuritiesCode', 'Date']).reset_index(drop=True)
gc.collect()

print(f"\nИтоговый фрейм цен: {prices.shape[0]:,} строк, "
      f"{prices['Date'].nunique()} торговых дней, {prices['SecuritiesCode'].nunique()} бумаг")

# Ограничиваем историю для расчёта признаков: на скрытых данных файл может оказаться
# заметно больше публичного, а нам нужны только последние годы рынка
MAX_TRAIN_DAYS = 1500
all_days = np.sort(prices['Date'].unique())
if len(all_days) > MAX_TRAIN_DAYS:
    prices = prices[prices['Date'].isin(all_days[-MAX_TRAIN_DAYS:])].reset_index(drop=True)
    all_days = np.sort(prices['Date'].unique())
    print(f"История ограничена последними {MAX_TRAIN_DAYS} днями: {len(prices):,} строк")

# Параметры walk-forward валидации
N_FOLDS = 4        # количество фолдов
FOLD_DAYS = 20     # дней в одном фолде
TRAIN_DAYS = 400   # размер обучающего окна (в днях)

# Подстройка под реальную длину истории: на скрытых данных дней может быть больше или меньше
while len(all_days) < N_FOLDS * FOLD_DAYS + TRAIN_DAYS:
    if TRAIN_DAYS > 60:
        TRAIN_DAYS //= 2
    elif FOLD_DAYS > 5:
        FOLD_DAYS //= 2
    else:
        N_FOLDS = max(1, N_FOLDS // 2)

CV_DAYS = N_FOLDS * FOLD_DAYS
val_all_days = all_days[-CV_DAYS:]                            # последние CV_DAYS дней — только валидация
train_all_days = all_days[-(CV_DAYS + TRAIN_DAYS):-CV_DAYS]   # окно, из которого берём обучение

print(f"Обучающее окно : {pd.Timestamp(train_all_days[0]).date()} — {pd.Timestamp(train_all_days[-1]).date()} ({len(train_all_days)} дн.)")
print(f"Валидация      : {pd.Timestamp(val_all_days[0]).date()} — {pd.Timestamp(val_all_days[-1]).date()} ({len(val_all_days)} дн.)")
print(f"Дней в фолде   : {FOLD_DAYS}, фолдов: {N_FOLDS}")

## 5. Создание признаков (Feature Engineering)

Признаки считаются функцией `add_features` на всём доступном прошлом. Ряды отсортированы по
`(SecuritiesCode, Date)`, все окна считаются внутри бумаги, заглядывания в будущее нет:
самый свежий признак для дня `t` использует данные строго до конца дня `t`.

Категории признаков:

* **доходности и лаги** — `ret_1/5/20/60`, гэп между открытием и предыдущим закрытием;
* **волатильность** — скользящие std доходностей, z-score объёма;
* **положение цены относительно SMA/EMA** — трендовые признаки;
* **свечной паттерн** — тело, диапазон, верхняя и нижняя тени;
* **рыночные (кросс-секционные)** — средняя доходность дня, избыточная доходность,
  ранг бумаги внутри дня. Такие признаки обычно сильнее всего работают в задаче ранжирования;
* **календарь и флаги** — день недели, месяц, наличие дивиденда, флаг наблюдения.

In [ ]:
print("Считаем признаки (отдельно по каждой бумаге)...")
t0 = time.time()
feat = add_features(prices)
print(f"Готово за {time.time() - t0:.1f} с.")

# бесконечности могли появиться при делении на нулевые SMA — приводим к NaN, ниже заполним нулём
feat = feat.replace([np.inf, -np.inf], np.nan)

print("\nДоля пропусков по признакам (топ-5):")
print(feat[FEATURES].isna().mean().sort_values(ascending=False).head(5).round(4).to_string())
print(f"В среднем пропусков на последних {CV_DAYS} днях: "
      f"{feat[feat['Date'].isin(val_all_days)][FEATURES].isna().mean().mean():.4%}")

print(f"\nПризнаков: {len(FEATURES)}, строк после feature engineering: {feat.shape[0]:,}")
print("Признаки:", FEATURES)

# Хвост истории пригодится в цикле по тестовым дням (раздел 10)
history_seed = (prices
                .sort_values(['SecuritiesCode', 'Date'])
                .groupby('SecuritiesCode', sort=False)
                .tail(HISTORY_DAYS)
                .reset_index(drop=True))
print(f"Хвост истории для теста: {len(history_seed):,} строк")

del prices
gc.collect()

## 6. Визуализация созданных признаков

In [ ]:
# Срез валидационных дней для визуализации
sample_data = feat[feat['Date'].isin(val_all_days)].reset_index(drop=True)
print("Срез для графиков:", sample_data.shape)

plot_features = ['body', 'hl_range', 'upper_shadow', 'lower_shadow', 'vol_20']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, f in enumerate(plot_features):
    axes[i].hist(sample_data[f].dropna(), bins=40, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Распределение: {f}')
    axes[i].set_xlabel(f)
    axes[i].set_ylabel('Частота')

axes[-1].hist(sample_data['ret_1'].dropna(), bins=60, edgecolor='black', alpha=0.7, color='green')
axes[-1].set_title('Распределение дневной доходности (ret_1)')
axes[-1].set_xlabel('ret_1')
axes[-1].set_ylabel('Частота')

plt.tight_layout()
plt.show()

In [ ]:
# Сравнение цены и скользящих средних для одной акции
last_security = int(history_seed['SecuritiesCode'].iloc[0])
sample_stock = feat[(feat['SecuritiesCode'] == last_security) & (feat['Date'].isin(all_days[-250:]))]
sample_stock = sample_stock.sort_values('Date')

fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(sample_stock['Date'], sample_stock['Close'], label='Close', color='black', linewidth=1.5)
ax.plot(sample_stock['Date'], sample_stock['px_sma20'] + 1, label='SMA 20 (Close/SMA-1 + 1)', linestyle='--', alpha=0.8)
ax.plot(sample_stock['Date'], sample_stock['px_sma60'] + 1, label='SMA 60 (Close/SMA-1 + 1)', linestyle='--', alpha=0.8)
ax.plot(sample_stock['Date'], sample_stock['px_ema20'] + 1, label='EMA 20 (Close/EMA-1 + 1)', linestyle=':', alpha=0.9)

ax.set_title(f'Цена и скользящие средние: бумага {last_security}')
ax.set_xlabel('Дата')
ax.set_ylabel('Цена (нормированная на 1)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Модели: линейный бейзлайн и градиентный бустинг

Задача — ранжирование, поэтому качество честнее мерить **дневным спредом доходностей**
(top-200 в лонг, bottom-200 в шорт), а не MSE: MSE почти не зависит от ранга и плохо
отражает то, что реально оценивает Kaggle.

Сравниваем две модели на одинаковых фолдах:

| Модель | Зачем |
|---|---|
| **Ridge** (после StandardScaler) | простой линейный бейзлайн — показывает, сколько даёт сама фактура признаков без нелинейностей |
| **LightGBM** | градиентный бустинг по деревьям — основная рабочая модель |

Каждый фолд обучается только на данных **до** начала фолда (walk-forward, утечки нет).

In [ ]:
BOOST_NAME = 'LightGBM' if LGBM_AVAILABLE else 'HistGB'
MODEL_NAMES = ['Ridge', BOOST_NAME]


def make_ridge(alpha=10.0):
    """Линейный бейзлайн: стандартизация + гребень."""
    return Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])


def make_boost():
    """
    Градиентный бустинг: LightGBM, при его отсутствии — HistGradientBoosting,
    а если нет и его — GradientBoosting (совместимость со старыми версиями sklearn).
    """
    if LGBM_AVAILABLE:
        return lgb.LGBMRegressor(
            objective='regression',
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.7,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
    if HISTGB_AVAILABLE:
        return HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=42)
    return GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, random_state=42)


def mean_daily_ic(df):
    """Средний ежедневный ранговый IC (Spearman) между предсказанием и Target."""
    return pd.Series(
        {day: group['pred'].corr(group['Target'], method='spearman')
         for day, group in df.groupby('Date', sort=True)},
        dtype='float64').mean()


def evaluate_predictions(df, portfolio_size=PORTFOLIO_SIZE):
    """
    df: колонки Date, SecuritiesCode, pred, Target.
    Возвращает Sharpe, Series дневных спред-доходностей и датафрейм с ранговым процентилем.

    ВАЖНО: ранговый процентиль считаем с ascending=True (по возрастанию), тогда у лучшей
    акции процентиль близок к 1. С ascending=True=False pandas наоборот присвоил бы
    лучшей акции минимальный процентиль, и любой блендинг по процентилям вышел бы инвертированным.
    """
    d = df.copy()
    d['Rank'] = (d.groupby('Date')['pred'].rank(ascending=False, method='first') - 1).astype(int)
    sharpe, buf = calc_spread_return_sharpe(d, portfolio_size=portfolio_size)
    d['pct'] = d.groupby('Date')['pred'].rank(pct=True)
    return sharpe, buf, d


fold_log = []
daily_spread = {name: [] for name in MODEL_NAMES}
scored_parts = {name: [] for name in MODEL_NAMES}
fold_results = pd.DataFrame(columns=['Фолд', 'Модель', 'Sharpe', 'IC', 'Дней', 'Время, с'])
CV_OK = True

# Валидация обёрнута в try/except: при отправке Kaggle перезапускает ноутбук на скрытых
# данных, и любая непредвиденная особенность этих данных не должна ронять всю отправку —
# модель и submission должны быть построены в любом случае.
try:
    for fold in range(N_FOLDS):
        val_days = val_all_days[fold * FOLD_DAYS:(fold + 1) * FOLD_DAYS]
        fold_start = len(all_days) - CV_DAYS + fold * FOLD_DAYS
        tr_days = all_days[max(0, fold_start - TRAIN_DAYS):fold_start]

        tr = feat[feat['Date'].isin(tr_days)].dropna(subset=[TARGET])
        va = feat[feat['Date'].isin(val_days)].dropna(subset=[TARGET])

        Xtr = tr[FEATURES].fillna(0).to_numpy(np.float32)
        ytr = tr[TARGET].to_numpy(np.float32)
        Xva = va[FEATURES].fillna(0).to_numpy(np.float32)

        print(f"\n=== Фолд {fold + 1}/{N_FOLDS}: обучение "
              f"{pd.Timestamp(tr_days[0]).date()}—{pd.Timestamp(tr_days[-1]).date()} ({len(tr):,} строк), "
              f"валидация {pd.Timestamp(val_days[0]).date()}—{pd.Timestamp(val_days[-1]).date()} ===")

        for name in MODEL_NAMES:
            t0 = time.time()
            model = make_ridge() if name == 'Ridge' else make_boost()
            model.fit(Xtr, ytr)
            pred = model.predict(Xva)

            fold_df = pd.DataFrame({
                'Date': va['Date'].to_numpy(),
                'SecuritiesCode': va['SecuritiesCode'].to_numpy(),
                'pred': pred,
                'Target': va[TARGET].to_numpy(),
            })
            sharpe, buf, scored = evaluate_predictions(fold_df)

            daily_spread[name].append(buf)
            scored_parts[name].append(scored[['Date', 'SecuritiesCode', 'pct', 'Target']])
            fold_log.append({
                'Фолд': fold + 1,
                'Модель': name,
                'Sharpe': round(float(sharpe), 3),
                'IC': round(float(mean_daily_ic(fold_df)), 4),
                'Дней': len(buf),
                'Время, с': round(time.time() - t0, 1),
            })

    fold_results = pd.DataFrame(fold_log)
    print("\n=== Sharpe Ratio и средний ранговый IC по фолдам ===")
    print(fold_results.pivot(index='Фолд', columns='Модель', values='Sharpe').to_string())
    print()
    print(fold_results.pivot(index='Фолд', columns='Модель', values='IC').to_string())

except Exception as exc:
    CV_OK = False
    print("\n[ВНИМАНИЕ] Кросс-валидация не завершилась: "
          f"{type(exc).__name__}: {exc}")
    print("Переходим к обучению финальной модели на всей доступной истории.")

In [ ]:
# ---- Сводим все фолды вместе и сравниваем модели целиком --------------------
scored = {name: (pd.concat(parts, ignore_index=True) if parts
                 else pd.DataFrame(columns=['Date', 'SecuritiesCode', 'pct', 'Target']))
          for name, parts in scored_parts.items()}
spread = {name: (pd.concat(daily_spread[name]) if daily_spread[name]
                 else pd.Series(dtype='float64'))
          for name in MODEL_NAMES}

# Блендинг: усредняем не предсказания, а ранговые процентили внутри дня —
# так модели с разными шкалами складываются корректно
blend = scored[BOOST_NAME][['Date', 'SecuritiesCode', 'pct', 'Target']].rename(columns={'pct': 'pct_boost'})
other = scored['Ridge'][['Date', 'SecuritiesCode', 'pct']].rename(columns={'pct': 'pct_ridge'})
blend = blend.merge(other, on=['Date', 'SecuritiesCode'], how='left')
blend['pred'] = blend[['pct_boost', 'pct_ridge']].mean(axis=1)

blend_sharpe, blend_buf, blend_scored = evaluate_predictions(blend[['Date', 'SecuritiesCode', 'pred', 'Target']])
scored['Blend'] = blend_scored[['Date', 'SecuritiesCode', 'pct', 'Target']]
spread['Blend'] = blend_buf
print(f"\nБлендинг рангов {BOOST_NAME} + Ridge: Sharpe = {blend_sharpe:.3f}")


def safe_sharpe(series):
    if len(series) < 2:
        return np.nan
    std = series.std()
    return float(series.mean() / std) if std and std > 0 else np.nan


summary = pd.DataFrame([
    {'Модель': name,
     'Sharpe (CV)': round(safe_sharpe(spread[name]), 3),
     'Средний спред': round(float(spread[name].mean()), 5) if len(spread[name]) else np.nan,
     'Ст. откл. спреда': round(float(spread[name].std()), 5) if len(spread[name]) else np.nan,
     'Дней': len(spread[name])}
    for name in spread
]).sort_values('Sharpe (CV)', ascending=False, na_position='last')

print("\n=== Итог по всей валидации (walk-forward) ===")
print(summary.to_string(index=False))

BEST_MODEL = summary.iloc[0]['Модель'] if len(summary) else BOOST_NAME
print(f"\nЛучшая модель по кросс-валидации: {BEST_MODEL} "
      f"(Sharpe = {summary.iloc[0]['Sharpe (CV)'] if len(summary) else 'нет данных'})")

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(summary['Модель'], summary['Sharpe (CV)'].fillna(0),
       color=['steelblue', 'darkorange', 'seagreen'][:len(summary)])
for i, v in enumerate(summary['Sharpe (CV)']):
    ax.text(i, (0 if pd.isna(v) else v) + 0.05, f'{v:.3f}', ha='center', va='bottom')
ax.axhline(0, color='red', linestyle='--', alpha=0.6)
ax.set_title('Sharpe Ratio на walk-forward валидации')
ax.set_ylabel('Sharpe Ratio')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Важность признаков -----------------------------------------------------
# Обучаем модели на последнем обучающем окне и смотрим, на что они смотрят
last_train_days = all_days[max(0, len(all_days) - CV_DAYS - TRAIN_DAYS):len(all_days) - CV_DAYS]
last_train = feat[feat['Date'].isin(last_train_days)].dropna(subset=[TARGET])
X_last = last_train[FEATURES].fillna(0).to_numpy(np.float32)
y_last = last_train[TARGET].to_numpy(np.float32)

ridge_last = make_ridge().fit(X_last, y_last)
ridge_imp = pd.Series(np.abs(ridge_last.named_steps['ridge'].coef_), index=FEATURES).sort_values(ascending=False)

boost_last = make_boost()
boost_imp = None
if LGBM_AVAILABLE:
    try:
        boost_last.fit(X_last, y_last, importance_type='gain')
        boost_imp = pd.Series(boost_last.feature_importances_, index=FEATURES).sort_values(ascending=False)
    except TypeError:
        # В старых версиях LightGBM параметр importance_type в fit() не поддерживается
        boost_last.fit(X_last, y_last)
        boost_imp = pd.Series(boost_last.feature_importances_, index=FEATURES).sort_values(ascending=False)
else:
    from sklearn.inspection import permutation_importance
    perm = permutation_importance(boost_last, X_last, y_last, n_repeats=3,
                                  random_state=42, scoring='neg_mean_squared_error',
                                  n_jobs=-1)
    boost_imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)

panels = [(f'{BOOST_NAME}: важность (gain)' if BOOST_NAME == 'LightGBM'
           else f'{BOOST_NAME}: permutation importance', boost_imp),
         ('Ridge: модуль коэффициента', ridge_imp)]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (title, imp) in zip(axes, panels):
    top = imp.head(15).sort_values()
    ax.barh(top.index, top.values, color='steelblue' if BOOST_NAME in title else 'darkorange')
    ax.set_title(title)
    ax.set_xlabel('важность')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Топ-15 признаков {BOOST_NAME}:")
print(boost_imp.head(15).round(4).to_string())
print("\nТоп-15 признаков Ridge:")
print(ridge_imp.head(15).round(4).to_string())

## 8. Статистическая проверка гипотез

Sharpe — выборочная статистика, поэтому проверим, что результат не случаен:

* **H₀: средняя дневная спред-доходность равна нулю** → t-тест по всем дням валидации
  (и Wilcoxon как непараметрическая проверка устойчивости);
* **H₀: средний ежедневный ранговый IC (Spearman между предсказанием и Target) равен нулю**
  → t-тест по IC;
* **попарное сравнение моделей** → связанный t-тест (`ttest_rel`) на дневных спредах.

Если p-value заметно меньше 0.05, положительный результат сложно объяснить случайностью —
при условии, что валидация построена walk-forward (раздел 4).

In [ ]:
def spearman_ic(df):
    """Ежедневный ранговый IC и t-тест против нуля."""
    ic = pd.Series(
        {day: group['pred'].corr(group['Target'], method='spearman')
         for day, group in df.groupby('Date', sort=True)},
        dtype='float64').dropna()
    t_stat, p_value = stats.ttest_1samp(ic, 0.0)
    return ic, t_stat, p_value


hyp_rows = []
for name in summary['Модель']:
    s = spread[name]
    if len(s) < 2:
        hyp_rows.append({'Модель': name, 'Sharpe': np.nan, 't (спред)': np.nan, 'p (спред)': np.nan,
                         'p (Wilcoxon)': np.nan, 'средний IC': np.nan, 't (IC)': np.nan, 'p (IC)': np.nan})
        continue

    t_stat, p_value = stats.ttest_1samp(s, 0.0)
    try:
        w_stat, w_p = stats.wilcoxon(s)
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    pred_df = scored[name][['Date', 'SecuritiesCode', 'pct', 'Target']].rename(columns={'pct': 'pred'})
    ic, ic_t, ic_p = spearman_ic(pred_df)

    hyp_rows.append({
        'Модель': name,
        'Sharpe': round(float(s.mean() / s.std()), 3),
        't (спред)': round(float(t_stat), 2),
        'p (спред)': f'{p_value:.2e}',
        'p (Wilcoxon)': f'{w_p:.2e}',
        'средний IC': round(float(ic.mean()), 4),
        't (IC)': round(float(ic_t), 2),
        'p (IC)': f'{ic_p:.2e}',
    })

hyp = pd.DataFrame(hyp_rows).sort_values('Sharpe', ascending=False, na_position='last')
print("=== Статистические тесты по валидации (уровень значимости 0.05) ===")
print(hyp.to_string(index=False))

if len(spread) > 1 and len(spread.get(BEST_MODEL, [])) > 1:
    for other_name in spread:
        if other_name == BEST_MODEL:
            continue
        if len(spread[other_name]) != len(spread[BEST_MODEL]):
            continue
        paired = stats.ttest_rel(spread[BEST_MODEL], spread[other_name])
        print(f"\nСвязанный t-тест  {BEST_MODEL} vs {other_name}: "
              f"t = {paired.statistic:.2f}, p-value = {paired.pvalue:.3f}")

## 9. Финальная модель

Обучаем выбранную по кросс-валидации комбинацию на максимально свежем окне
(`FINAL_TRAIN_DAYS` последних торговых дней). Модель фиксируется **до** старта итерации
по тестовым дням — это соответствует правилам соревнования (никакого peek в будущее) и
экономит время: переобучать бустинг каждый день не требуется.

In [ ]:
FINAL_TRAIN_DAYS = 600
final_days = all_days[-min(FINAL_TRAIN_DAYS, len(all_days)):]
final_df = feat[feat['Date'].isin(final_days)].dropna(subset=[TARGET])

X_final = final_df[FEATURES].fillna(0).to_numpy(np.float32)
y_final = final_df[TARGET].to_numpy(np.float32)

print(f"Финальное обучение: {pd.Timestamp(final_days[0]).date()} — {pd.Timestamp(final_days[-1]).date()}, "
      f"{len(final_df):,} строк")

MODEL_READY = True
ridge_final = None
boost_final = None

try:
    t0 = time.time()
    ridge_final = make_ridge().fit(X_final, y_final)
    print(f"Ridge обучен за {time.time() - t0:.1f} с.")

    t0 = time.time()
    boost_final = make_boost()
    boost_final.fit(X_final, y_final)
    print(f"{BOOST_NAME} обучен за {time.time() - t0:.1f} с.")
except Exception as exc:
    MODEL_READY = False
    print(f"[ВНИМАНИЕ] Финальные модели не обучились ({type(exc).__name__}: {exc}). "
          "Будет использован запасной предиктор по последней дневной доходности.")


def predict_scores(X, dates):
    """
    Итоговый счёт для строк X (одна или несколько дат).

    Модели выдают шкалы разного масштаба, поэтому для блендинга используются
    ранговые процентили внутри дня, а для одиночной модели — её собственный прогноз.
    """
    day = pd.Series(dates).to_numpy()

    if not MODEL_READY:
        # Запасной сценарий: ранг по последней дневной доходности (простой импульс).
        # Лучше слабый, но осмысленный прогноз, чем падение всей отправки.
        return pd.Series(X[:, FEATURES.index('ret_1')]).groupby(day).rank(pct=True).to_numpy()

    if BEST_MODEL == 'Ridge':
        return ridge_final.predict(X)
    if BEST_MODEL == BOOST_NAME:
        return boost_final.predict(X)

    # Blend: среднее ранговых процентилей двух моделей внутри каждого дня
    # (процентиль считаем по возрастанию: у лучшей акции он близок к 1)
    pct_ridge = pd.Series(ridge_final.predict(X)).groupby(day).rank(pct=True).to_numpy()
    pct_boost = pd.Series(boost_final.predict(X)).groupby(day).rank(pct=True).to_numpy()
    return 0.5 * (pct_ridge + pct_boost)


print(f"\nФинальный предиктор: {BEST_MODEL if MODEL_READY else 'запасной (импульс)'}")

## 10. Генерация `submission.csv` через официальный time-series API

**Почему в первой версии была ошибка.** В этом соревновании файл `submission.csv` создаёт
сам API: он накапливает результаты `env.predict(...)` и в конце итерации записывает их в
`submission.csv`. Kaggle закрывает этот путь, поэтому `submission_*.to_csv('submission.csv')`
заканчивается `PermissionError: [Errno 1] Operation not permitted: 'submission.csv'`.
Ноутбук при этом отрабатывал целиком — падала только самая последняя ячейка, поэтому
локально (где папка writable) всё выглядело исправным.

**Официальный шаблон соревнования:**

```python
import jpx_tokyo_market_prediction
env = jpx_tokyo_market_prediction.make_env()
iter_test = env.iter_test()
for (prices, options, financials, trades, secondary_prices, sample_prediction) in iter_test:
    sample_prediction['Rank'] = ...   # ранг на текущий день
    env.predict(sample_prediction)    # регистрируем прогноз
```

Что нужно соблюдать:

* ранг строго от `0` до `N-1` для каждого дня, **без дубликатов**;
* **порядок строк** `sample_prediction` менять нельзя — заполняем только колонку `Rank`;
* данные приходят по одному дню, поэтому историю котировок нужно доносить вручную —
  иначе лаги и скользящие окна «слепнут».

Вне Kaggle (например, на локальной машине) пакет API недоступен — там используется
`local_iter_test()`, которая повторяет формат API на файлах `example_test_files`.

In [ ]:
PRICE_COLS = ['Open', 'High', 'Low', 'Close', 'Volume', 'ExpectedDividend', 'AdjustmentFactor']


def normalize_prices(df):
    """Приводит котировки, пришедшие из API, к тому же виду, что и при обучении."""
    out = df.copy()
    if 'RowId' in out.columns:
        out = out.drop(columns=['RowId'])
    out['Date'] = pd.to_datetime(out['Date'])
    out['SecuritiesCode'] = out['SecuritiesCode'].astype('int32')
    for c in PRICE_COLS:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors='coerce').astype('float32')
    if 'SupervisionFlag' in out.columns:
        try:
            out['SupervisionFlag'] = out['SupervisionFlag'].astype('boolean').fillna(False).astype('bool')
        except (TypeError, ValueError):
            out['SupervisionFlag'] = False
    else:
        out['SupervisionFlag'] = False
    return out[['Date', 'SecuritiesCode'] + PRICE_COLS + ['SupervisionFlag']]


def keep_last_days(df, n_days):
    """Последние n_days наблюдений по каждой бумаге — скользящее окно для признаков."""
    return (df.sort_values(['SecuritiesCode', 'Date'])
              .groupby('SecuritiesCode', sort=False)
              .tail(n_days)
              .reset_index(drop=True))


def local_iter_test(data_dir=None):
    """
    Локальная имитация официального API на файлах example_test_files.
    Используется только вне Kaggle (на Kaggle работает настоящий API).
    """
    data_dir = data_dir or os.path.join(DATA_DIR, 'example_test_files')
    names = {
        'prices': 'stock_prices.csv',
        'options': 'options.csv',
        'financials': 'financials.csv',
        'trades': 'trades.csv',
        'secondary_prices': 'secondary_stock_prices.csv',
    }
    tables = {}
    for key, fname in names.items():
        path = os.path.join(data_dir, fname)
        tables[key] = pd.read_csv(path) if os.path.exists(path) else None

    sample = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))
    prices_local = tables['prices']

    def day_slice(table, day):
        return table[table['Date'] == day].reset_index(drop=True) if table is not None else None

    for day in sorted(prices_local['Date'].unique()):
        yield (day_slice(tables['prices'], day),
               day_slice(tables['options'], day),
               day_slice(tables['financials'], day),
               day_slice(tables['trades'], day),
               day_slice(tables['secondary_prices'], day),
               day_slice(sample, day))


history = normalize_prices(history_seed)
del history_seed
gc.collect()
print(f"История перед стартом теста: {len(history):,} строк, {history['Date'].nunique()} дней")

local_predictions = []
if API_AVAILABLE:
    env = jpx_tokyo_market_prediction.make_env()
    iter_test = env.iter_test()
    register_prediction = env.predict
    print("Используется официальный time-series API Kaggle.")
elif ON_KAGGLE:
    # Молча уходить в локальную имитацию нельзя: тогда Kaggle не найдёт submission.csv
    # и отклонит отправку с сообщением «the selected Notebook Version does not output this file».
    raise RuntimeError(
        "На Kaggle не удалось подключить официальный time-series API, поэтому submission.csv "
        "создан не будет — этот файл формирует сам API.\n\n"
        f"Версия Python : {sys.version.split()[0]}\n"
        f"Ошибка импорта: {API_IMPORT_ERROR}\n\n"
        "Причина: пакет jpx_tokyo_market_prediction собран под CPython 3.7 "
        "(competition.cpython-37m-x86_64-linux-gnu.so) и не импортируется в более новом Python.\n\n"
        "Что нужно сделать:\n"
        "  1) НЕ загружать .ipynb через Upload — это даёт обычный ноутбук на Python 3.13;\n"
        "  2) открыть страницу соревнования -> Code -> New Notebook\n"
        "     (получится Competition Notebook с окружением Python 3.7 и подключёнными данными);\n"
        "  3) перенести в него код из этого ноутбука\n"
        "     (в загруженном ноутбуке: ... -> Import notebook -> Add to notebook);\n"
        "  4) Run all и убедиться, что в выводе: Python: 3.7.x и 'Официальный API доступен: True';\n"
        "  5) Save Version -> Submit to competition."
    )
else:
    iter_test = local_iter_test()
    register_prediction = local_predictions.append
    print("Локальный запуск (не Kaggle) — используется local_iter_test().")

In [ ]:
n_days = 0
n_fallback_days = 0

for (day_prices, options, financials, trades, secondary_prices, sample_prediction) in iter_test:
    t0 = time.time()
    day = None

    try:
        # 1) Доносим историю: на случай повторного дня выкидываем старые строки этой даты
        day_prices = normalize_prices(day_prices)
        day = day_prices['Date'].iloc[0]
        history = history[history['Date'] != day]
        history = keep_last_days(pd.concat([history, day_prices], ignore_index=True), HISTORY_DAYS)

        # 2) Признаки на текущий день
        feats = add_features(history)
        feats_day = feats[feats['Date'] == day].copy()
        del feats
        gc.collect()

        # 3) Прогноз и плотные ранги внутри дня
        X = feats_day[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(np.float32)
        preds = predict_scores(X, feats_day['Date'].to_numpy())

        rank_df = pd.DataFrame({
            'Date': feats_day['Date'].to_numpy(),
            'SecuritiesCode': feats_day['SecuritiesCode'].to_numpy(),
            'pred': preds,
        })
        rank_df['Rank'] = (rank_df.groupby('Date')['pred']
                           .rank(ascending=False, method='first') - 1).astype(int)

        # 4) Заполняем Rank в шаблоне, сохраняя исходный порядок строк
        rank_map = {(d, c): int(r) for d, c, r
                    in zip(rank_df['Date'], rank_df['SecuritiesCode'], rank_df['Rank'])}
        sub_dates = pd.to_datetime(sample_prediction['Date'])
        sample_prediction['Rank'] = [rank_map.get((d, c), -1)
                                     for d, c in zip(sub_dates, sample_prediction['SecuritiesCode'])]

        # бумаги, которых не было в котировках дня, получают свободные ранги с конца
        missing = sample_prediction['Rank'] < 0
        if missing.any():
            free = sorted(set(range(len(sample_prediction))) - set(sample_prediction.loc[~missing, 'Rank']))
            sample_prediction.loc[missing, 'Rank'] = free[:int(missing.sum())]

    except Exception as exc:
        # Мягкая деградация: лучше отдать «средний» прогноз, чем вообще не отдать submission.
        # Так одна проблемная итерация не убьёт весь перезапуск ноутбука на скрытых данных.
        n_fallback_days += 1
        print(f"[ВНИМАНИЕ] День {day} обработан запасным сценарием: {type(exc).__name__}: {exc}")
        sample_prediction['Rank'] = np.arange(len(sample_prediction))

    # 5) Проверки требований API (в любом случае ранг должен быть плотным и без дублей)
    sample_prediction['Rank'] = pd.to_numeric(sample_prediction['Rank'], errors='coerce')
    if sample_prediction['Rank'].isna().any() or not sample_prediction['Rank'].is_unique \
            or sample_prediction['Rank'].min() != 0 \
            or sample_prediction['Rank'].max() != len(sample_prediction) - 1:
        n_fallback_days += 1
        print("[ВНИМАНИЕ] Ранги не прошли проверку — подставляем нейтральный порядок.")
        sample_prediction['Rank'] = np.arange(len(sample_prediction))

    sample_prediction['Rank'] = sample_prediction['Rank'].astype(int)

    try:
        register_prediction(sample_prediction)
    except Exception as exc:
        # Даже отказ API не должен обрывать отправку: последняя попытка — нейтральный порядок
        n_fallback_days += 1
        print(f"[ВНИМАНИЕ] API отклонил прогноз за день {day} "
              f"({type(exc).__name__}: {exc}); пробуем нейтральный порядок.")
        sample_prediction['Rank'] = np.arange(len(sample_prediction))
        register_prediction(sample_prediction)

    n_days += 1
    if n_days <= 3 or n_days % 10 == 0:
        print(f"День {day.date() if day is not None else '?'}: строк {len(sample_prediction):,}, "
              f"уникальных рангов {sample_prediction['Rank'].nunique()}, {time.time() - t0:.1f} с")

print(f"\nОбработано дней: {n_days}, из них с запасным сценарием: {n_fallback_days}")

In [ ]:
if API_AVAILABLE:
    print("Готово: submission.csv сформирован официальным API (вручную этот файл не пишется).")
else:
    # Локальный файл для проверки. ВАЖНО: это не submission.csv —
    # на Kaggle такой файл создаёт API, а ручная запись в submission.csv даёт PermissionError.
    local_sub = pd.concat(local_predictions, ignore_index=True)
    local_sub.to_csv('local_submission.csv', index=False)
    print(f"Локальная проверка: local_submission.csv {local_sub.shape}, "
          f"дней {local_sub['Date'].nunique()}")